# 02 — Settlement Observability Diagnostics

This notebook evaluates settlement-scale observability of daily VIIRS Black Marble VNP46A2 nighttime lights across GHSL-SMOD classes in Samar–Leyte. It extends the POI-based diagnostics by moving from local pixel supports to settlement-structured aggregation.

The analysis compares direct DNB-BRDF radiance, spatial completeness, temporal completeness, dropout duration, and variability across the urban–rural settlement hierarchy. This provides the settlement-scale basis for determining which land-cover and settlement supports are most interpretable for reliability-qualified NTL analysis.

---

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

## 1. Input Data and Output Paths

### 1.1 Required input

This notebook uses daily class-level VNP46A2 DNB-BRDF statistics stratified by GHSL-SMOD class for Samar–Leyte.

| Input                                      | Repository path                                                                                  | Description                                   |
| ------------------------------------------ | ------------------------------------------------------------------------------------------------ | --------------------------------------------- |
| GHSL class-level daily DNB-BRDF statistics | `datasets/vnp46a2/settlement_daily_stats/SamarLeyte_DNBBRDF_AllClasses_DailyStats_2012_2024.csv` | Daily DNB-BRDF statistics by GHSL-SMOD class. |

Expected input columns:

| Column        | Description                                                               |
| ------------- | ------------------------------------------------------------------------- |
| `date`        | Observation date.                                                         |
| `ClassCode`   | GHSL-SMOD settlement class code.                                          |
| `NTL_mean`    | Mean DNB-BRDF radiance.                                                   |
| `NTL_median`  | Median DNB-BRDF radiance.                                                 |
| `NTL_min`     | Minimum DNB-BRDF radiance.                                                |
| `NTL_max`     | Maximum DNB-BRDF radiance.                                                |
| `NTL_std`     | Standard deviation of DNB-BRDF radiance.                                  |
| `Valid_px`    | Number of valid observed pixels for the class and date.                   |
| `Mean_by_sum` | Alternative class-level mean radiance metric used for visual diagnostics. |

In [ ]:
# ==============================
# Output Paths
# ==============================

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "vnp46a2"
    / "settlement_daily_stats"
    / "SamarLeyte_DNBBRDF_AllClasses_DailyStats_2012_2024.csv"
)

# Main manuscript outputs
OUTPUT_MAIN_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "main"
OUTPUT_MAIN_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "main"

# Supplementary outputs
OUTPUT_SUPP_FIG_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
    / "supplementary"
    / "settlement_observability"
)

OUTPUT_SUPP_TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "supplementary"
    / "settlement_observability"
)

for out_dir in [
    OUTPUT_MAIN_FIG_DIR,
    OUTPUT_MAIN_TABLE_DIR,
    OUTPUT_SUPP_FIG_DIR,
    OUTPUT_SUPP_TABLE_DIR,
]:
    out_dir.mkdir(parents=True, exist_ok=True)

print(f"Main figures: {OUTPUT_MAIN_FIG_DIR}")
print(f"Main tables: {OUTPUT_MAIN_TABLE_DIR}")
print(f"Supplementary figures: {OUTPUT_SUPP_FIG_DIR}")
print(f"Supplementary tables: {OUTPUT_SUPP_TABLE_DIR}")

In [ ]:
def make_plotly_export_safe(obj):
    """Convert pandas/numpy objects before Plotly static export."""
    if isinstance(obj, pd.Timestamp):
        return obj.to_pydatetime()

    if isinstance(obj, np.datetime64):
        return pd.Timestamp(obj).to_pydatetime()

    if isinstance(obj, pd.DatetimeIndex):
        return obj.to_pydatetime().tolist()

    if isinstance(obj, np.ndarray):
        return [make_plotly_export_safe(v) for v in obj.tolist()]

    if isinstance(obj, list):
        return [make_plotly_export_safe(v) for v in obj]

    if isinstance(obj, tuple):
        return tuple(make_plotly_export_safe(v) for v in obj)

    if isinstance(obj, dict):
        return {k: make_plotly_export_safe(v) for k, v in obj.items()}

    return obj


def export_plotly_figure(
    fig,
    output_path,
    width=1400,
    height=800,
    scale=3,
    export_png=True,
    export_svg=True,
    export_html=True,
):
    """
    Export Plotly figures to high-resolution PNG/SVG and optional HTML.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig_safe = go.Figure(make_plotly_export_safe(fig.to_plotly_json()))

    if export_png:
        fig_safe.write_image(
            str(output_path.with_suffix(".png")),
            width=width,
            height=height,
            scale=scale,
        )

    if export_svg:
        fig_safe.write_image(
            str(output_path.with_suffix(".svg")),
            width=width,
            height=height,
            scale=scale,
        )

    if export_html:
        fig_safe.write_html(
            str(output_path.with_suffix(".html")),
            include_plotlyjs="cdn",
            full_html=True,
        )

    print(f"Exported base path: {output_path}")

## 2. Load GHSL-Stratified Daily Statistics

### 2.1 Load and prepare daily class-level table

This section loads the GHSL-stratified daily DNB-BRDF statistics and prepares the 2013 subset used for Typhoon Haiyan visual diagnostics.

The full 2012–2024 table is retained for long-term observability and variability diagnostics.


In [ ]:
df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])
df["ClassCode"] = df["ClassCode"].astype(str)

df = df.sort_values(["ClassCode", "date"]).copy()

df_2013 = df[df["date"].dt.year == 2013].copy()
df_2013 = df_2013.sort_values(["ClassCode", "date"])

df_2013.head()

In [ ]:
input_check_path = OUTPUT_SUPP_TABLE_DIR / "ghsl_class_daily_stats_2013_head.csv"
df_2013.head(20).to_csv(input_check_path, index=False)

print(f"Exported: {input_check_path}")

## 3. GHSL-SMOD Class Metadata

### 3.1 Settlement class definitions

The GHSL-SMOD classes are used to organise the NTL signal along a settlement-intensity gradient, from water and rural grid cells to dense urban centres.

| Class code | Label                    | Interpretation                     |
| ---------- | ------------------------ | ---------------------------------- |
| 10         | Water                    | Water bodies.                      |
| 11         | Very low-density rural   | Sparse rural grid cells.           |
| 12         | Low-density rural        | Low-density rural grid cells.      |
| 13         | Rural cluster            | Rural clustered settlement.        |
| 21         | Suburban / peri-urban    | Peri-urban or suburban settlement. |
| 22         | Semi-dense urban cluster | Intermediate urban cluster.        |
| 23         | Dense urban cluster      | Dense urban built-up area.         |
| 30         | Urban centre             | High-density urban centre.         |

This hierarchy is central to the reliability framework because nighttime-light signals are not equally interpretable across all settlement contexts. Dense built-up classes carry stronger anthropogenic lighting signals, while rural and water classes have lower baseline radiance and greater proportional sensitivity to noise.


In [ ]:
GHSL_INFO = {
    "10": {"label": "Water (10)", "color": "#7ab6f5"},
    "11": {"label": "Very Low Density Rural (11)", "color": "#cdf57a"},
    "12": {"label": "Low Density Rural (12)", "color": "#abcd66"},
    "13": {"label": "Rural Cluster (13)", "color": "#375623"},
    "21": {"label": "Suburban / Peri-urban (21)", "color": "#ffff00"},
    "22": {"label": "Semi-dense Urban Cluster (22)", "color": "#a87000"},
    "23": {"label": "Dense Urban Cluster (23)", "color": "#732600"},
    "30": {"label": "Urban Centre (30)", "color": "#ff0000"},
}

GHSL_ORDER = sorted(GHSL_INFO.keys(), key=int)

GHSL_LABELS = {
    cls: GHSL_INFO[cls]["label"]
    for cls in GHSL_ORDER
}

GHSL_COLORS = {
    GHSL_INFO[cls]["label"]: GHSL_INFO[cls]["color"]
    for cls in GHSL_ORDER
}

ghsl_metadata = pd.DataFrame([
    {
        "ClassCode": cls,
        "ClassLabel": GHSL_INFO[cls]["label"],
        "Color": GHSL_INFO[cls]["color"],
    }
    for cls in GHSL_ORDER
])

ghsl_metadata

In [ ]:
# ==============================
# Export: Main Table 1
# GHSL-SMOD Class Metadata
# ==============================

ghsl_metadata_main = pd.DataFrame([
    {
        "Code": cls,
        "Official Class": GHSL_INFO[cls]["label"].replace(f" ({cls})", ""),
        "Analysis Group": CLASS_TO_GROUP.get(cls, np.nan),
    }
    for cls in GHSL_ORDER
])

ghsl_metadata_main_path = OUTPUT_MAIN_TABLE_DIR / "table_01_ghsl_smod_classes_and_analysis_groups.csv"
ghsl_metadata_main.to_csv(ghsl_metadata_main_path, index=False)

print(f"Exported main table: {ghsl_metadata_main_path}")

## 4. Shared Plotting Utilities

### 4.1 Valid-pixel filtering and event marker

The visual diagnostics apply a percentile filter and valid-pixel threshold to reduce the influence of extreme outliers and very low-support observations. This keeps the figures focused on class-level structure rather than isolated anomalous values.

The threshold is used for visual diagnostics only. The later reliability table reports multiple temporal-completeness thresholds more explicitly.


In [ ]:
HAIYAN_DATE = pd.Timestamp("2013-11-08").to_pydatetime()

def add_haiyan_marker(fig, label="Haiyan"):
    """Add a consistent Typhoon Haiyan marker to Plotly figures."""
    fig.add_vline(
        x=HAIYAN_DATE,
        line_dash="dash",
        line_color="black",
        line_width=3,
    )

    fig.add_annotation(
        x=HAIYAN_DATE,
        y=1.05,
        yref="paper",
        text=label,
        showarrow=False,
        font=dict(size=16, color="black"),
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        xanchor="left",
    )

    return fig


def prepare_class_plot_data(
    df_input,
    value_col="Mean_by_sum",
    threshold=10,
    lower_percentile=0,
    upper_percentile=95,
):
    """
    Prepare GHSL class-level data for visual diagnostics.

    Applies:
    - class-wise percentile filtering
    - class-wise spatial completeness calculation
    - valid-pixel threshold filtering
    """
    plot_frames = []

    df_plot = df_input.copy()
    df_plot["date"] = pd.to_datetime(df_plot["date"])
    df_plot["ClassCode"] = df_plot["ClassCode"].astype(str)

    for cls in GHSL_ORDER:
        df_cls = df_plot[df_plot["ClassCode"] == cls].copy()

        if df_cls.empty:
            continue

        if df_cls[value_col].notna().sum() > 0:
            p_low = np.nanpercentile(df_cls[value_col], lower_percentile)
            p_high = np.nanpercentile(df_cls[value_col], upper_percentile)

            df_cls = df_cls[
                (df_cls[value_col] >= p_low)
                & (df_cls[value_col] <= p_high)
            ].copy()

        total_px = df_cls["Valid_px"].max()

        if total_px and total_px > 0:
            df_cls["Valid_pct"] = df_cls["Valid_px"] / total_px * 100
            df_cls = df_cls[df_cls["Valid_pct"] > threshold].copy()

        if df_cls.empty:
            continue

        df_cls["ClassLabel"] = df_cls["ClassCode"].map(GHSL_LABELS)
        df_cls["date_plot"] = pd.to_datetime(df_cls["date"]).dt.to_pydatetime()

        plot_frames.append(df_cls)

    if not plot_frames:
        return pd.DataFrame()

    return pd.concat(plot_frames, ignore_index=True)

## 5. Daily Scatter by GHSL Class

### 5.1 Raw class-level daily points, 2013

This figure shows daily class-level DNB-BRDF radiance values for 2013 after percentile and valid-pixel filtering. Each point represents one daily observation for a GHSL-SMOD class.

The plot is intentionally unsmoothed to expose dispersion, class separation, and the abruptness of the Haiyan-period radiance disruption.


In [ ]:
def plot_timeseries_by_class_points(
    df_input,
    value_col="Mean_by_sum",
    threshold=10,
):
    df_plot = prepare_class_plot_data(
        df_input,
        value_col=value_col,
        threshold=threshold,
        lower_percentile=0,
        upper_percentile=95,
    )

    fig = go.Figure()

    for cls in GHSL_ORDER:
        label = GHSL_INFO[cls]["label"]
        color = GHSL_INFO[cls]["color"]

        df_cls = df_plot[df_plot["ClassCode"] == cls].copy()

        if df_cls.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=df_cls["date_plot"],
                y=df_cls[value_col],
                mode="markers",
                marker=dict(
                    color=color,
                    size=6,
                    opacity=0.6,
                ),
                name=label,
            )
        )

    add_haiyan_marker(fig)

    fig.update_layout(
        xaxis=dict(
            title="Date",
            title_font=dict(size=20),
            tickfont=dict(size=16),
        ),
        yaxis=dict(
            title="Radiance (nW·cm⁻²·sr⁻¹)",
            title_font=dict(size=20),
            tickfont=dict(size=16),
        ),
        legend=dict(
            orientation="h",
            y=1.2,
            x=0.02,
            xanchor="left",
            font=dict(size=14),
        ),
        template="plotly_white",
        height=500,
        width=1300,
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig, df_plot


fig_scatter, df_scatter_plot = plot_timeseries_by_class_points(
    df_2013,
    value_col="Mean_by_sum",
    threshold=10,
)

fig_scatter.show()

In [ ]:
# ==============================
# Export: Supplementary Figure
# GHSL Class Daily Scatter, 2013
# ==============================

scatter_data_path = (
    OUTPUT_SUPP_TABLE_DIR
    / "fig_sx_ghsl_class_daily_scatter_2013_plot_data.csv"
)

df_scatter_plot.to_csv(scatter_data_path, index=False)

export_plotly_figure(
    fig_scatter,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_ghsl_class_daily_scatter_2013",
    width=1300,
    height=500,
    scale=3,
)

print(f"Exported supplementary plot data: {scatter_data_path}")

### 5.2 Plot discussion

The scatter plot shows clear radiance stratification by settlement intensity. Urban Centre has the highest radiance values and the greatest dispersion, while dense and semi-dense urban clusters form an intermediate tier. Rural and water classes remain near lower radiance levels.

The Haiyan marker highlights that the strongest structural radiance collapse occurs in the urban classes. Rural and water classes show much weaker event-scale change, consistent with their lower anthropogenic lighting contribution. This supports the manuscript argument that settlement structure conditions the interpretability of daily NTL signals.


## 6. Class-Wise Time Series

### 6.1 Class-level trajectories with minimal smoothing

This figure plots class-level DNB-BRDF trajectories after valid-pixel filtering. The line uses a rolling window parameter, but a window of `w = 1` preserves daily variability.

This plot is used to visualise settlement-scale temporal structure while retaining the event-period signal.

In [ ]:
def plot_timeseries_by_class(
    df_input,
    value_col="Mean_by_sum",
    w=1,
    threshold=40,
):
    df_plot = prepare_class_plot_data(
        df_input,
        value_col=value_col,
        threshold=threshold,
        lower_percentile=0,
        upper_percentile=95,
    )

    fig = go.Figure()

    for cls in GHSL_ORDER:
        label = GHSL_INFO[cls]["label"]
        color = GHSL_INFO[cls]["color"]

        df_cls = df_plot[df_plot["ClassCode"] == cls].copy()

        if df_cls.empty:
            continue

        df_cls = df_cls.sort_values("date").copy()
        df_cls[f"MA_{w}d"] = (
            df_cls[value_col]
            .rolling(w, min_periods=1)
            .mean()
        )

        fig.add_trace(
            go.Scatter(
                x=df_cls["date_plot"],
                y=df_cls[value_col],
                mode="markers",
                marker=dict(size=6, color=color, opacity=0.45),
                showlegend=False,
            )
        )

        fig.add_trace(
            go.Scatter(
                x=df_cls["date_plot"],
                y=df_cls[f"MA_{w}d"],
                mode="lines",
                line=dict(color=color, width=2),
                name=label,
            )
        )

    add_haiyan_marker(fig)

    fig.update_layout(
        xaxis=dict(
            title="Date",
            title_font=dict(size=20),
            tickfont=dict(size=16),
        ),
        yaxis=dict(
            title="Radiance (nW·cm⁻²·sr⁻¹)",
            title_font=dict(size=20),
            tickfont=dict(size=16),
        ),
        legend=dict(
            orientation="h",
            y=1.2,
            x=0.05,
            xanchor="left",
            font=dict(size=14),
        ),
        template="plotly_white",
        height=500,
        width=1300,
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig, df_plot


fig_class_ts, df_class_ts_plot = plot_timeseries_by_class(
    df_2013,
    value_col="Mean_by_sum",
    w=1,
    threshold=40,
)

fig_class_ts.show()

In [ ]:
# ==============================
# Export: Supplementary Figure
# GHSL Class Time Series, 2013
# ==============================

class_ts_data_path = (
    OUTPUT_SUPP_TABLE_DIR
    / "fig_sx_ghsl_class_timeseries_2013_threshold40_plot_data.csv"
)

df_class_ts_plot.to_csv(class_ts_data_path, index=False)

export_plotly_figure(
    fig_class_ts,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_ghsl_class_timeseries_2013_threshold40",
    width=1300,
    height=500,
    scale=3,
)

print(f"Exported supplementary plot data: {class_ts_data_path}")

### 6.2 Plot discussion

The time-series figure shows that settlement intensity structures both baseline radiance and disaster response. Urban Centre and dense urban classes exhibit the clearest pre-event lighting signal and the most visible post-Haiyan collapse. Rural and water classes remain lower and less event-responsive.

The figure reinforces why GHSL-based settlement stratification is necessary. Without separating settlement classes, the regional mean may mix interpretable urban outage signals with low-signal rural or water pixels, weakening the physical meaning of the NTL trajectory.


## 7. Settlement-Scale Reliability Diagnostics

### 7.1 Metric definitions

This section computes settlement-scale observability and variability metrics for each GHSL-SMOD class.

Metrics are computed for the 2013 event year to align with the Haiyan-focused visual diagnostics.

| Metric                     | Description                                                                  |
| -------------------------- | ---------------------------------------------------------------------------- |
| Spatial completeness       | Mean daily valid-pixel share within the class.                               |
| Temporal completeness ≥25% | Fraction of days with at least 25% valid pixels.                             |
| Temporal completeness ≥50% | Fraction of days with at least 50% valid pixels.                             |
| Temporal completeness ≥75% | Fraction of days with at least 75% valid pixels.                             |
| Maximum dropout            | Longest consecutive run of days below the 50% valid-pixel threshold.         |
| Temporal CV                | Variability of daily mean radiance relative to its mean.                     |
| Spatial CV                 | Median within-class spatial variability, calculated as `NTL_std / NTL_mean`. |

Umbrella groups are used to compare broader settlement supports:

| Group            | GHSL classes |
| ---------------- | ------------ |
| Urban Centre     | 30           |
| Urban Cluster    | 21, 22, 23   |
| Rural Grid Cells | 11, 12, 13   |
| Water Bodies     | 10           |


In [ ]:
ANALYSIS_START = "2013-01-01"
ANALYSIS_END = "2013-12-31"
DATE_RANGE = pd.date_range(ANALYSIS_START, ANALYSIS_END, freq="D")

PIXEL_DICT = {
    "10": {"count": 834.0, "pct": 0.99},
    "11": {"count": 53864.0, "pct": 63.78},
    "12": {"count": 13773.0, "pct": 16.39},
    "13": {"count": 3364.0, "pct": 3.98},
    "21": {"count": 9406.0, "pct": 11.14},
    "22": {"count": 1322.0, "pct": 1.57},
    "23": {"count": 1418.0, "pct": 1.68},
    "30": {"count": 467.0, "pct": 0.55},
}

UMBRELLA_MAP = {
    "Urban Centre": ["30"],
    "Urban Cluster": ["21", "22", "23"],
    "Rural Grid Cells": ["13", "12", "11"],
    "Water Bodies": ["10"],
}

CLASS_TO_GROUP = {
    cls: group
    for group, classes in UMBRELLA_MAP.items()
    for cls in classes
}

GROUP_ORDER = ["Urban Centre", "Urban Cluster", "Rural Grid Cells", "Water Bodies"]

CLASS_ORDER_WITHIN = {
    "Urban Centre": ["30"],
    "Urban Cluster": ["21", "22", "23"],
    "Rural Grid Cells": ["13", "12", "11"],
    "Water Bodies": ["10"],
}

In [ ]:
# ==============================
# Code edit: per-class reliability metrics
# ==============================

df_metric = df[
    (df["date"] >= pd.Timestamp(ANALYSIS_START))
    & (df["date"] <= pd.Timestamp(ANALYSIS_END))
].copy()

df_metric["ClassCode"] = df_metric["ClassCode"].astype(str)

rows = []

for cls, sub in df_metric.groupby("ClassCode"):
    px_info = PIXEL_DICT.get(cls, {"count": np.nan, "pct": np.nan})
    pixel_count = px_info["count"]

    s = (
        sub.set_index("date")
        .reindex(DATE_RANGE)
        .rename_axis("date")
        .reset_index()
    )

    for col in ["Valid_px", "NTL_mean", "NTL_std"]:
        if col not in s.columns:
            s[col] = np.nan

    s[["Valid_px", "NTL_mean", "NTL_std"]] = (
        s[["Valid_px", "NTL_mean", "NTL_std"]].fillna(0)
    )

    spatial = (
        (s["Valid_px"] / pixel_count).mean() * 100
        if pixel_count > 0
        else np.nan
    )

    temporal_25 = (s["Valid_px"] >= pixel_count * 0.25).mean() * 100
    temporal_50 = (s["Valid_px"] >= pixel_count * 0.50).mean() * 100
    temporal_75 = (s["Valid_px"] >= pixel_count * 0.75).mean() * 100

    meets_50 = (s["Valid_px"] >= pixel_count * 0.50).astype(int)
    grp = (meets_50 != meets_50.shift()).cumsum()
    dropout_lengths = meets_50[meets_50.eq(0)].groupby(grp[meets_50.eq(0)]).size()
    max_dropout = int(dropout_lengths.max()) if not dropout_lengths.empty else 0

    v = s[(s["Valid_px"] > 0) & (s["NTL_mean"] > 0)].copy()

    temporal_cv = (
        v["NTL_mean"].std() / v["NTL_mean"].mean()
        if not v.empty and v["NTL_mean"].mean() > 0
        else np.nan
    )

    spatial_cv = (
        (v["NTL_std"] / v["NTL_mean"]).replace([np.inf, -np.inf], np.nan).median()
        if not v.empty
        else np.nan
    )

    rows.append({
        "Group": CLASS_TO_GROUP.get(cls),
        "ClassCode": cls,
        "ClassLabel": GHSL_LABELS.get(cls, f"Class {cls}"),
        "Pixel_count": pixel_count,
        "Pixel_share(%)": px_info["pct"],
        "Spatial_completeness(%)": spatial,
        "Temporal_completeness_ge25(%)": temporal_25,
        "Temporal_completeness_ge50(%)": temporal_50,
        "Temporal_completeness_ge75(%)": temporal_75,
        "Max_dropout_days": max_dropout,
        "Temporal_CV": temporal_cv,
        "Spatial_CV": spatial_cv,
    })

class_stats = pd.DataFrame(rows)
class_stats

In [ ]:
avg_rows = []

for group in GROUP_ORDER:
    sub = class_stats[class_stats["Group"] == group].copy()

    if sub.empty:
        continue

    avg_rows.append({
        "Group": f"{group} (average)",
        "ClassCode": "avg",
        "ClassLabel": f"{group} (average)",
        "Pixel_count": sub["Pixel_count"].sum(),
        "Pixel_share(%)": sub["Pixel_share(%)"].sum(),
        "Spatial_completeness(%)": sub["Spatial_completeness(%)"].mean(),
        "Temporal_completeness_ge25(%)": sub["Temporal_completeness_ge25(%)"].mean(),
        "Temporal_completeness_ge50(%)": sub["Temporal_completeness_ge50(%)"].mean(),
        "Temporal_completeness_ge75(%)": sub["Temporal_completeness_ge75(%)"].mean(),
        "Max_dropout_days": sub["Max_dropout_days"].max(),
        "Temporal_CV": sub["Temporal_CV"].mean(),
        "Spatial_CV": sub["Spatial_CV"].mean(),
    })

group_stats = pd.DataFrame(avg_rows)

summary = pd.concat([group_stats, class_stats], ignore_index=True)

def settlement_order_key(row):
    group_base = row["Group"].replace(" (average)", "")

    return (
        GROUP_ORDER.index(group_base),
        0 if row["ClassCode"] == "avg" else 1,
        CLASS_ORDER_WITHIN.get(group_base, []).index(row["ClassCode"])
        if row["ClassCode"] in CLASS_ORDER_WITHIN.get(group_base, [])
        else 99,
    )

summary["order_key"] = summary.apply(settlement_order_key, axis=1)

summary = (
    summary.sort_values("order_key")
    .drop(columns="order_key")
    .reset_index(drop=True)
)

summary = summary.round({
    "Pixel_count": 0,
    "Pixel_share(%)": 2,
    "Spatial_completeness(%)": 1,
    "Temporal_completeness_ge25(%)": 1,
    "Temporal_completeness_ge50(%)": 1,
    "Temporal_completeness_ge75(%)": 1,
    "Temporal_CV": 2,
    "Spatial_CV": 2,
})

summary

In [ ]:
# ==============================
# Export: Supplementary Full Settlement Metrics
# ==============================

supp_table_s3 = summary.copy()

supp_table_s3_path = (
    OUTPUT_SUPP_TABLE_DIR
    / "table_s3_full_settlement_observability_metrics_2013.csv"
)

supp_table_s3.to_csv(supp_table_s3_path, index=False)

print(f"Exported supplementary table: {supp_table_s3_path}")

In [ ]:
# ==============================
# Export: Main Table 3
# Settlement-Scale Observability Diagnostics
# ==============================

main_table_03 = group_stats.copy()

main_table_03["Settlement Group"] = (
    main_table_03["Group"]
    .str.replace(" \\(average\\)", "", regex=True)
)

main_table_03 = main_table_03[
    [
        "Settlement Group",
        "Pixel_share(%)",
        "Spatial_completeness(%)",
        "Temporal_completeness_ge25(%)",
        "Temporal_completeness_ge50(%)",
        "Temporal_completeness_ge75(%)",
        "Max_dropout_days",
        "Temporal_CV",
        "Spatial_CV",
    ]
].copy()

main_table_03 = main_table_03.rename(columns={
    "Pixel_share(%)": "Pixel Share (%)",
    "Spatial_completeness(%)": "Spatial Completeness (%)",
    "Temporal_completeness_ge25(%)": "Temporal Completeness ≥25% (%)",
    "Temporal_completeness_ge50(%)": "Temporal Completeness ≥50% (%)",
    "Temporal_completeness_ge75(%)": "Temporal Completeness ≥75% (%)",
    "Max_dropout_days": "Max Dropout Days",
    "Temporal_CV": "Temporal CV",
    "Spatial_CV": "Spatial CV",
})

main_table_03 = main_table_03.round({
    "Pixel Share (%)": 2,
    "Spatial Completeness (%)": 1,
    "Temporal Completeness ≥25% (%)": 1,
    "Temporal Completeness ≥50% (%)": 1,
    "Temporal Completeness ≥75% (%)": 1,
    "Temporal CV": 2,
    "Spatial CV": 2,
})

main_table_03_path = OUTPUT_MAIN_TABLE_DIR / "table_03_settlement_observability_diagnostics_2013.csv"
main_table_03.to_csv(main_table_03_path, index=False)

print(f"Exported main table: {main_table_03_path}")

main_table_03

### 7.2 Table discussion

The reliability summary shows that spatial completeness is broadly similar across land-based settlement classes, with urban, peri-urban, and rural classes all affected by the same tropical cloud regime. This supports the manuscript framing that cloud-driven observability constraints are not confined to one settlement type.

However, interpretability differs across classes. Urban Centre and Urban Cluster classes have stronger anthropogenic lighting signals, making radiance declines more meaningful for electricity disruption analysis. Rural grid cells and water bodies have weaker or less settlement-specific signals, so comparable spatial completeness does not imply comparable interpretability.

The maximum dropout values show that multi-week gaps can occur even at settlement-aggregated scales. This is important because aggregation can increase the number of valid pixels on observed days, but it cannot fully remove temporal gaps caused by persistent cloud cover.


## 8. Annual Variability Diagnostics

### 8.1 Annual coefficient of variation from daily DNB-BRDF

This section evaluates year-to-year radiance variability by GHSL class across the full 2012–2024 record. The annual coefficient of variation is calculated from daily class-level `NTL_mean` values within each year.

These diagnostics quantify whether settlement classes differ not only in brightness, but also in signal stability.


In [ ]:
annual_cv_raw = (
    df[df["NTL_mean"] > 0]
    .assign(
        year=df["date"].dt.year,
        ClassCode=df["ClassCode"].astype(str),
    )
    .groupby(["ClassCode", "year"])
    .agg(
        mean_ntl=("NTL_mean", "mean"),
        std_ntl=("NTL_mean", "std"),
    )
    .reset_index()
)

annual_cv_raw["CV"] = annual_cv_raw["std_ntl"] / annual_cv_raw["mean_ntl"]
annual_cv_raw["ClassLabel"] = annual_cv_raw["ClassCode"].map(GHSL_LABELS)

annual_cv_raw = annual_cv_raw.dropna(subset=["ClassLabel", "CV"])
annual_cv_raw.head()

In [ ]:
ordered_labels = [GHSL_INFO[k]["label"] for k in GHSL_ORDER]

fig_cv_line_raw = px.line(
    annual_cv_raw,
    x="year",
    y="CV",
    color="ClassLabel",
    color_discrete_map=GHSL_COLORS,
    category_orders={"ClassLabel": ordered_labels},
)

fig_cv_line_raw.update_traces(mode="lines+markers", line=dict(width=2))

fig_cv_line_raw.update_layout(
    template="plotly_white",
    width=1100,
    height=500,
    legend=dict(
        orientation="h",
        y=1.18,
        x=0.5,
        xanchor="center",
        font=dict(size=14),
    ),
    xaxis=dict(
        title="Year",
        title_font=dict(size=20),
        tickfont=dict(size=16),
    ),
    yaxis=dict(
        title="NTL coefficient of variation",
        type="log",
        title_font=dict(size=20),
        tickfont=dict(size=16),
    ),
    paper_bgcolor="rgba(0,0,0,0)",
)

fig_cv_line_raw.show()

In [ ]:
# ==============================
# Export: Supplementary Figure
# Annual CV from Raw Daily NTL
# ==============================

annual_cv_raw_path = (
    OUTPUT_SUPP_TABLE_DIR
    / "annual_cv_raw_daily_ntl_by_ghsl_class_2012_2024.csv"
)

annual_cv_raw.to_csv(annual_cv_raw_path, index=False)

export_plotly_figure(
    fig_cv_line_raw,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_annual_cv_raw_daily_ntl_by_ghsl_class_2012_2024",
    width=1100,
    height=500,
    scale=3,
)

print(f"Exported supplementary table: {annual_cv_raw_path}")

In [ ]:
fig_cv_box_raw = px.box(
    annual_cv_raw,
    x="ClassLabel",
    y="CV",
    color="ClassLabel",
    color_discrete_map=GHSL_COLORS,
    category_orders={"ClassLabel": ordered_labels},
    points="outliers",
)

fig_cv_box_raw.update_layout(
    template="plotly_white",
    width=1100,
    height=550,
    showlegend=False,
    xaxis=dict(
        title="GHSL-SMOD class",
        title_font=dict(size=20),
        tickfont=dict(size=13),
    ),
    yaxis=dict(
        title="NTL coefficient of variation",
        type="log",
        title_font=dict(size=20),
        tickfont=dict(size=16),
    ),
    paper_bgcolor="rgba(0,0,0,0)",
)

medians_raw = annual_cv_raw.groupby("ClassLabel")["CV"].median()

for cls, val in medians_raw.items():
    fig_cv_box_raw.add_annotation(
        x=cls,
        y=val,
        text=f"{val:.2f}",
        showarrow=False,
        font=dict(size=13),
        yshift=-30,
    )

fig_cv_box_raw.show()

In [ ]:
# ==============================
# Export: Supplementary Figure
# Raw Annual CV Distribution
# ==============================

export_plotly_figure(
    fig_cv_box_raw,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_annual_cv_distribution_raw_daily_ntl_by_ghsl_class_2012_2024",
    width=1100,
    height=550,
    scale=3,
)

### 8.2 Plot discussion

The annual CV diagnostics show that settlement classes differ in signal stability as well as mean radiance. Urban classes generally carry stronger and more interpretable radiance signals, but they can still show substantial variability due to disaster shocks, recovery dynamics, land-use heterogeneity, and observation gaps.

Low-light classes can show high proportional variability because small absolute changes are amplified when the mean radiance is close to background levels. This is why CV must be interpreted alongside class brightness and settlement meaning rather than as a standalone reliability score.


## 9. MA30-Based Annual Variability Diagnostics

### 9.1 Thirty-day moving average CV

This section repeats the annual CV analysis after applying a 30-day moving average to class-level daily `NTL_mean`. The smoothed series reduces day-to-day noise and highlights broader seasonal or structural variability.

The MA30-based CV is more aligned with the later NTL–load alignment workflow, where smoothed trajectories are used to reduce high-frequency noise before comparing against electricity load.

In [ ]:
df_ma = df.sort_values(["ClassCode", "date"]).copy()

df_ma["NTL_MA30"] = (
    df_ma.groupby("ClassCode")["NTL_mean"]
    .transform(lambda x: x.rolling(30, min_periods=1).mean())
)

annual_cv_ma30 = (
    df_ma[df_ma["NTL_MA30"] > 0]
    .assign(
        year=df_ma["date"].dt.year,
        ClassCode=df_ma["ClassCode"].astype(str),
    )
    .groupby(["ClassCode", "year"])
    .agg(
        mean_ntl=("NTL_MA30", "mean"),
        std_ntl=("NTL_MA30", "std"),
    )
    .reset_index()
)

annual_cv_ma30["CV"] = annual_cv_ma30["std_ntl"] / annual_cv_ma30["mean_ntl"]
annual_cv_ma30["ClassLabel"] = annual_cv_ma30["ClassCode"].map(GHSL_LABELS)

annual_cv_ma30 = annual_cv_ma30.dropna(subset=["ClassLabel", "CV"])
annual_cv_ma30.head()

In [ ]:

fig_cv_line_ma30 = px.line(
    annual_cv_ma30,
    x="year",
    y="CV",
    color="ClassLabel",
    color_discrete_map=GHSL_COLORS,
    category_orders={"ClassLabel": ordered_labels},
)

fig_cv_line_ma30.update_traces(mode="lines+markers", line=dict(width=2))

fig_cv_line_ma30.update_layout(
    template="plotly_white",
    width=1100,
    height=500,
    legend=dict(
        orientation="h",
        y=1.18,
        x=0.5,
        xanchor="center",
        font=dict(size=14),
    ),
    xaxis=dict(
        title="Year",
        title_font=dict(size=20),
        tickfont=dict(size=16),
    ),
    yaxis=dict(
        title="NTL coefficient of variation (MA30)",
        type="log",
        title_font=dict(size=20),
        tickfont=dict(size=16),
    ),
    paper_bgcolor="rgba(0,0,0,0)",
)

fig_cv_line_ma30.show()

In [ ]:
# ==============================
# Export: Supplementary Figure
# Annual CV from MA30 NTL
# ==============================

annual_cv_ma30_path = (
    OUTPUT_SUPP_TABLE_DIR
    / "annual_cv_ma30_ntl_by_ghsl_class_2012_2024.csv"
)

annual_cv_ma30.to_csv(annual_cv_ma30_path, index=False)

export_plotly_figure(
    fig_cv_line_ma30,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_annual_cv_ma30_ntl_by_ghsl_class_2012_2024",
    width=1100,
    height=500,
    scale=3,
)

print(f"Exported supplementary table: {annual_cv_ma30_path}")

In [ ]:
fig_cv_box_ma30 = px.box(
    annual_cv_ma30,
    x="ClassLabel",
    y="CV",
    color="ClassLabel",
    color_discrete_map=GHSL_COLORS,
    category_orders={"ClassLabel": ordered_labels},
    points="outliers",
)

fig_cv_box_ma30.update_layout(
    template="plotly_white",
    width=1100,
    height=550,
    showlegend=False,
    xaxis=dict(
        title="GHSL-SMOD class",
        title_font=dict(size=20),
        tickfont=dict(size=13),
    ),
    yaxis=dict(
        title="NTL coefficient of variation (MA30)",
        type="log",
        title_font=dict(size=20),
        tickfont=dict(size=16),
    ),
    paper_bgcolor="rgba(0,0,0,0)",
)

medians_ma30 = annual_cv_ma30.groupby("ClassLabel")["CV"].median()

for cls, val in medians_ma30.items():
    fig_cv_box_ma30.add_annotation(
        x=cls,
        y=val,
        text=f"{val:.2f}",
        showarrow=False,
        font=dict(size=13),
        yshift=-30,
    )

fig_cv_box_ma30.show()

In [ ]:
# ==============================
# Export: Supplementary Figure
# Annual CV from MA30 NTL
# ==============================

annual_cv_ma30_path = (
    OUTPUT_SUPP_TABLE_DIR
    / "annual_cv_ma30_ntl_by_ghsl_class_2012_2024.csv"
)

annual_cv_ma30.to_csv(annual_cv_ma30_path, index=False)

export_plotly_figure(
    fig_cv_line_ma30,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_annual_cv_ma30_ntl_by_ghsl_class_2012_2024",
    width=1100,
    height=500,
    scale=3,
)

print(f"Exported supplementary table: {annual_cv_ma30_path}")

### 9.2 Plot discussion

The MA30-based CV diagnostics reduce short-term noise and highlight class-level structural stability. Settlement classes with persistent anthropogenic lighting remain more interpretable because their radiance signals are anchored by stronger baseline illumination. Rural and water classes can remain proportionally volatile because low baseline radiance magnifies relative variation.

This supports the manuscript’s reliability framing: settlement aggregation can improve interpretability, but only when paired with observability thresholds and class-aware interpretation. A class can have moderate spatial completeness but still provide a weak recovery proxy if its radiance is low, unstable, or weakly connected to electricity demand.


## 10. Consolidated Variability Summary

### 10.1 Class-level CV comparison table

This table summarises median annual CV values from both the raw daily series and the MA30-smoothed series.


In [ ]:
cv_raw_summary = (
    annual_cv_raw.groupby(["ClassCode", "ClassLabel"])["CV"]
    .median()
    .reset_index()
    .rename(columns={"CV": "Median_CV_raw"})
)

cv_ma30_summary = (
    annual_cv_ma30.groupby(["ClassCode", "ClassLabel"])["CV"]
    .median()
    .reset_index()
    .rename(columns={"CV": "Median_CV_MA30"})
)

cv_summary = cv_raw_summary.merge(
    cv_ma30_summary,
    on=["ClassCode", "ClassLabel"],
    how="outer",
)

cv_summary["Group"] = cv_summary["ClassCode"].map(CLASS_TO_GROUP)

cv_summary = cv_summary[
    ["Group", "ClassCode", "ClassLabel", "Median_CV_raw", "Median_CV_MA30"]
].copy()

cv_summary = cv_summary.sort_values(
    "ClassCode",
    key=lambda s: s.astype(int),
).round(3)

cv_summary

In [ ]:
# ==============================
# Export: Supplementary Table
# Raw and MA30 CV Summary
# ==============================

cv_summary_path = (
    OUTPUT_SUPP_TABLE_DIR
    / "ghsl_class_cv_summary_raw_and_ma30_2012_2024.csv"
)

cv_summary.to_csv(cv_summary_path, index=False)

print(f"Exported supplementary table: {cv_summary_path}")

### 10.2 Interpretation

The consolidated CV table separates two types of instability. The raw CV reflects high-frequency daily variability, including abrupt observation changes and event-driven fluctuations. The MA30 CV reflects broader temporal instability after short-term noise is reduced.

Classes with consistently lower CV after smoothing are more stable candidates for downstream alignment. Classes that remain volatile after smoothing require stronger observability screening or should be treated cautiously in recovery interpretation.


## 11. Notebook Outputs

This notebook exports both main-manuscript and supplementary outputs.

### Main manuscript outputs

| Output | Path |
|---|---|
| Table 1 — GHSL-SMOD classes and analysis groups | `outputs/tables/main/table_01_ghsl_smod_classes_and_analysis_groups.csv` |
| Table 3 — Settlement-scale observability diagnostics | `outputs/tables/main/table_03_settlement_observability_diagnostics_2013.csv` |

### Supplementary outputs

| Output | Path |
|---|---|
| Full settlement observability metrics | `outputs/tables/supplementary/settlement_observability/table_s3_full_settlement_observability_metrics_2013.csv` |
| GHSL class scatter plot data | `outputs/tables/supplementary/settlement_observability/fig_sx_ghsl_class_daily_scatter_2013_plot_data.csv` |
| GHSL class time-series plot data | `outputs/tables/supplementary/settlement_observability/fig_sx_ghsl_class_timeseries_2013_threshold40_plot_data.csv` |
| Raw annual CV table | `outputs/tables/supplementary/settlement_observability/annual_cv_raw_daily_ntl_by_ghsl_class_2012_2024.csv` |
| MA30 annual CV table | `outputs/tables/supplementary/settlement_observability/annual_cv_ma30_ntl_by_ghsl_class_2012_2024.csv` |
| Raw and MA30 CV summary | `outputs/tables/supplementary/settlement_observability/ghsl_class_cv_summary_raw_and_ma30_2012_2024.csv` |
| GHSL class daily scatter figure | `outputs/figures/supplementary/settlement_observability/fig_sx_ghsl_class_daily_scatter_2013.*` |
| GHSL class time-series figure | `outputs/figures/supplementary/settlement_observability/fig_sx_ghsl_class_timeseries_2013_threshold40.*` |
| Raw annual CV line figure | `outputs/figures/supplementary/settlement_observability/fig_sx_annual_cv_raw_daily_ntl_by_ghsl_class_2012_2024.*` |
| Raw annual CV box figure | `outputs/figures/supplementary/settlement_observability/fig_sx_annual_cv_distribution_raw_daily_ntl_by_ghsl_class_2012_2024.*` |
| MA30 annual CV line figure | `outputs/figures/supplementary/settlement_observability/fig_sx_annual_cv_ma30_ntl_by_ghsl_class_2012_2024.*` |
| MA30 annual CV box figure | `outputs/figures/supplementary/settlement_observability/fig_sx_annual_cv_distribution_ma30_ntl_by_ghsl_class_2012_2024.*` |

## 12. Summary

This notebook shows that settlement structure strongly conditions the interpretability of daily DNB-BRDF nighttime lights. Urban and dense built-up classes provide stronger anthropogenic lighting signals and clearer event responses, while rural and water classes have lower baseline radiance and greater proportional sensitivity to noise.

At the same time, spatial completeness is not exclusively an urban or rural issue. Land-based settlement classes experience broadly similar cloud-driven observability constraints, including prolonged dropout periods. This means that settlement aggregation alone is insufficient; class selection must be paired with explicit spatial-completeness and temporal-completeness screening.

These results motivate the next notebook, which evaluates GHSL mask and spatial-completeness threshold combinations against independent electricity load. The key transition is from settlement observability to reliability-qualified NTL–load alignment.